In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)

tensor([[ 2.8328e-02, -7.3910e-02, -1.7003e-01,  5.4218e-01,  5.8821e-02,
         -2.4626e-01, -6.5322e-02,  1.1235e-01,  3.7604e-03,  5.8027e-02],
        [-3.7059e-04, -1.3191e-01, -2.3233e-01,  5.6370e-01, -1.5836e-02,
         -2.4276e-01, -6.4888e-03,  1.9784e-01, -4.6835e-02,  1.3256e-01]],
       grad_fn=<AddmmBackward0>)

In [2]:
class MLP(nn.Module):
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）
        super().__init__()
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.out = nn.Linear(256, 10)  # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义。
        return self.out(F.relu(self.hidden(X)))

In [3]:
net = MLP()
net(X)

tensor([[ 0.0078, -0.0074,  0.0821,  0.0940,  0.1341, -0.2069,  0.2117,  0.0234,
          0.1493, -0.4035],
        [ 0.0702, -0.0534,  0.1013,  0.1351,  0.0449, -0.1626,  0.2600, -0.0688,
          0.2896, -0.3042]], grad_fn=<AddmmBackward0>)

### 顺序块

In [8]:
class MySequential(nn.Module):
    def __init__(self, *args):  # *args是收集参数，相当于把若干个参数打包成一个来传入
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

In [9]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[ 0.2636, -0.1283, -0.1206,  0.0430,  0.0514, -0.2535,  0.2344, -0.0614,
          0.0663,  0.1392],
        [ 0.3361, -0.1835, -0.2207,  0.0451,  0.0763, -0.4428,  0.2526, -0.0572,
         -0.0053,  0.0464]], grad_fn=<AddmmBackward0>)

### 在正向传播函数中执行代码

In [8]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

In [9]:
net = FixedHiddenMLP()
net(X)

tensor(-0.2773, grad_fn=<SumBackward0>)

### 混合搭配各种组合块的方法


In [10]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(0.2071, grad_fn=<SumBackward0>)

In [11]:
### 通过继承 nn.module类来做比较灵活的模型构造